# Gold Layer - Customer Churn Risk Analysis

## Objective

The objective of this notebook is to identify customers who are at risk of churning based on their purchase recency.

This Gold table directly addresses one of the key business questions from the RetailMart problem statement:

**"Which customers are about to churn?"**

Since no explicit churn label exists in the dataset, customer churn risk is estimated using Recency Analysis.

The latest order date available in the dataset is treated as the reference date.

Customers are classified into different churn risk segments based on the number of days since their most recent purchase.

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
SOURCE1 = GOLD_FACT_SALES
SOURCE2 = GOLD_CUSTOMER_360
TARGET_TABLE = GOLD_CUSTOMER_CHURN

In [0]:
spark.table(GOLD_CUSTOMER_360).printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_purchased: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- average_item_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- customer_lifetime_days: integer (nullable = true)



In [0]:
spark.table(GOLD_FACT_SALES).printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_month: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- total_payment_value: double (nullable = true)



In [0]:
## Step 1: Determine Dataset Reference Date
reference_date_df = spark.sql(f"""
SELECT
MAX(order_purchase_timestamp) AS latest_order_date
FROM {SOURCE1}
""")
display(reference_date_df)

latest_order_date
2023-06-20T23:50:00.000Z


In [0]:
## Step 2: Build Customer Churn Risk Table
spark.sql(f"""

CREATE OR REPLACE TABLE {TARGET_TABLE}
USING DELTA
AS
WITH reference_date AS (
    SELECT
        MAX(order_purchase_timestamp) AS latest_order_date
    FROM {SOURCE1}
)

SELECT

    c.customer_id,
    c.customer_city,
    c.customer_state,

    c.total_orders,
    c.total_items_purchased,
    c.total_spent,
    c.average_item_value,

    c.first_purchase,
    c.last_purchase,
    c.customer_lifetime_days,

    DATEDIFF(
        r.latest_order_date,
        c.last_purchase
    ) AS days_since_last_purchase,

    CASE

        WHEN c.total_orders = 0
            THEN 'Never Purchased'

        WHEN DATEDIFF(r.latest_order_date, c.last_purchase) > 180
            THEN 'High Risk - Likely Churned'

        WHEN DATEDIFF(r.latest_order_date, c.last_purchase) > 90
            THEN 'Medium Risk'

        ELSE 'Active'

    END AS churn_risk_segment

FROM {SOURCE2} c

CROSS JOIN reference_date r

""")

print("Gold Customer Churn Risk table created successfully.")

Gold Customer Churn Risk table created successfully.


In [0]:
## Step 3: Validate Gold Table
customer_churn_df = spark.table(TARGET_TABLE)
display(customer_churn_df.limit(10))

customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days,days_since_last_purchase,churn_risk_segment
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809,0,Active
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415,425,High Risk - Likely Churned
CUST_000066,Campo Grande,MS,4,4,4156.87,1039.2175,2021-06-09T23:00:00.000Z,2022-06-14T08:46:00.000Z,370,371,High Risk - Likely Churned
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741,60,Active
CUST_000095,Teresina,PI,1,1,1036.53,1036.53,2022-08-08T15:08:00.000Z,2022-08-08T15:08:00.000Z,0,316,High Risk - Likely Churned
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559,325,High Risk - Likely Churned
CUST_000117,Manaus,AM,0,0,0.0,0.0,null,null,0,null,Never Purchased
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849,20,Active
CUST_000136,Joao Pessoa,PB,1,3,3487.1,1162.3666666666666,2021-06-08T11:29:00.000Z,2021-06-08T11:29:00.000Z,0,742,High Risk - Likely Churned
CUST_000159,Porto Velho,RO,2,2,2271.27,1135.635,2021-11-11T15:22:00.000Z,2022-06-21T01:37:00.000Z,222,364,High Risk - Likely Churned


In [0]:
customer_churn_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_purchased: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- average_item_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- customer_lifetime_days: integer (nullable = true)
 |-- days_since_last_purchase: integer (nullable = true)
 |-- churn_risk_segment: string (nullable = true)



In [0]:
customer_churn_df.describe().show()

+-------+-----------+-------------+--------------+------------------+---------------------+-----------------+------------------+----------------------+------------------------+------------------+
|summary|customer_id|customer_city|customer_state|      total_orders|total_items_purchased|      total_spent|average_item_value|customer_lifetime_days|days_since_last_purchase|churn_risk_segment|
+-------+-----------+-------------+--------------+------------------+---------------------+-----------------+------------------+----------------------+------------------------+------------------+
|  count|      15000|        15000|         15000|             15000|                15000|            15000|             15000|                 15000|                   14481|             15000|
|   mean|       NULL|         NULL|          NULL|3.3333333333333335|               5.7552|7486.081435999998|1257.2911443209643|    412.95366666666666|       235.2386575512741|              NULL|
| stddev|       NULL

In [0]:
## Step 4: Customer Churn Risk Distribution
display(

spark.sql(f"""
SELECT
    churn_risk_segment,
    COUNT(*) AS customer_count
FROM {TARGET_TABLE}
GROUP BY churn_risk_segment
ORDER BY customer_count DESC
""")

)

churn_risk_segment,customer_count
High Risk - Likely Churned,7144
Active,4379
Medium Risk,2958
Never Purchased,519


In [0]:
## Step 5: High Risk Customers
display(

spark.sql(f"""

SELECT
    customer_id,
    customer_city,
    customer_state,
    total_orders,
    total_spent,
    days_since_last_purchase,
    churn_risk_segment
FROM {TARGET_TABLE}
WHERE churn_risk_segment = 'High Risk - Likely Churned'
ORDER BY total_spent DESC
LIMIT 20
""")

)

customer_id,customer_city,customer_state,total_orders,total_spent,days_since_last_purchase,churn_risk_segment
CUST_005390,Sao Paulo,SP,7,29005.22,250,High Risk - Likely Churned
CUST_009393,Porto Alegre,RS,8,28999.46,297,High Risk - Likely Churned
CUST_003678,Joao Pessoa,PB,8,28673.0,309,High Risk - Likely Churned
CUST_010797,Porto Alegre,RS,9,28065.359999999997,251,High Risk - Likely Churned
CUST_011521,Curitiba,PR,8,26743.869999999995,200,High Risk - Likely Churned
CUST_005954,Curitiba,PR,8,26296.160000000003,247,High Risk - Likely Churned
CUST_013529,Curitiba,PR,8,25856.5,194,High Risk - Likely Churned
CUST_010595,Recife,PE,9,25760.73,190,High Risk - Likely Churned
CUST_001081,Fortaleza,CE,9,24839.800000000003,244,High Risk - Likely Churned
CUST_014346,Aracaju,SE,7,24459.550000000003,237,High Risk - Likely Churned


In [0]:
## Step 6: Gold Table Summary
print(f"Target Table : {TARGET_TABLE}")
print(f"Total Customers : {customer_churn_df.count()}")
print(f"Columns : {len(customer_churn_df.columns)}")

Target Table : retailmart.gold.customer_churn
Total Customers : 15000
Columns : 12


# Engineering Observations

- Built a Gold Layer table for customer churn risk analysis.
- Used the latest purchase date available in the dataset as the reference date.
- Calculated customer inactivity using the number of days since the last purchase.
- Classified customers into Active, Medium Risk, High Risk, and Never Purchased segments.
- The table directly addresses the RetailMart business question:
  **"Which customers are about to churn?"**
- The output is optimized for executive dashboards and customer retention analysis.